# Обучение модели для RiskScore
Модель строит два линейных прогноза для низкого и высокого риска. Ещё одна
линейная регрессия оценивает вес верхней ветки по признакам заявки.

In [10]:
import pathlib
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

root = Path.cwd()

## Подготовка выборок

Убираю строки с повреждённым `RiskScore`, точные дубли и строки с пропусками.
Идентификатор тестовой заявки понадобится при сохранении ответа.

In [11]:
raw = pd.read_csv(root / "train.csv")
valid_target = raw.RiskScore.notna() & ~raw.RiskScore.isin([-9999999, 10000000])
clean = raw.loc[valid_target].drop_duplicates().dropna().reset_index(drop=True)
y = clean.RiskScore.to_numpy()
train = clean.drop(columns="RiskScore")
test = pd.read_csv(root / "test.csv")
print(f"Строк для обучения: {len(train):,}".replace(",", " "))

Строк для обучения: 9 332


## Пороговые признаки

Каждая ступенька равна единице при достижении порога. Например, заявки с
`CreditScore` 620 и 720 проходят разное число порогов. Храню ступеньки
отдельными столбцами, чтобы модель могла выучить свой эффект для каждого
порога. Вместе с тремя бинарными флагами это 25 основных признаков.

In [12]:
MAIN_THRESHOLDS = {
    "CreditScore": [600, 650, 700, 750],
    "DebtToIncomeRatio": [0.2, 0.3, 0.4, 0.5],
    "CreditCardUtilizationRate": [0.2, 0.4, 0.6],
    "LengthOfCreditHistory": [3, 7, 10],
    "AnnualIncome": [30_000, 50_000, 80_000, 120_000],
    "NetWorth": [50_000, 100_000, 250_000, 500_000],
}


def threshold_features(frame, grids):
    return {
        f"{name}>={threshold}": frame[name].ge(threshold).astype(float)
        for name, thresholds in grids.items()
        for threshold in thresholds
    }


def main_features(frame):
    features = threshold_features(frame, MAIN_THRESHOLDS)
    features["Bankruptcy"] = frame.BankruptcyHistory.astype(float)
    features["PreviousDefaults"] = frame.PreviousLoanDefaults.astype(float)
    features["NotEmployed"] = frame.EmploymentStatus.ne("Employed").astype(float)
    return pd.DataFrame(features, index=frame.index)

## Дополнительные признаки

Добавляю пороги из прежнего решения, исключая повторения основной сетки.
Например, порог в 30 000 годового дохода уже есть в `MAIN_THRESHOLDS`.
Финансовые отношения описывают долговую нагрузку, капитал и кредитную
историю. Для них применяю знаковый логарифм
$\operatorname{sign}(x)\log(1+|x|)$, чтобы сжать длинные хвосты.

In [13]:
EXTRA_THRESHOLDS = {
    "CreditScore": [500, 550, 800, 850],
    "MonthlyIncome": np.array([15, 20, 40, 75, 100, 150, 200, 300]) * 1000 / 12,
    "DebtToIncomeRatio": [0.35, 0.6],
    "TotalDebtToIncomeRatio": [0.1, 0.2, 0.3, 0.4, 0.5, 1, 2],
    "NetWorth": [10_000, 25_000, 200_000, 1_000_000],
    "Age": [25, 30, 35, 40, 45, 50, 55, 60, 65],
    "CreditCardUtilizationRate": [0.1, 0.3, 0.5, 0.7, 0.8, 0.9],
    "LengthOfCreditHistory": [5, 15, 20, 25],
}


def financial_ratios(x):
    income = x.AnnualIncome.replace(0, np.nan)
    monthly = x.MonthlyIncome.replace(0, np.nan)
    return pd.DataFrame({
        "IncomePerDependent": x.AnnualIncome / (x.NumberOfDependents + 1).replace(0, np.nan),
        "DebtBurden": x.MonthlyDebtPayments / monthly,
        "MonthlyIncomePerDebt": x.MonthlyIncome / (x.MonthlyDebtPayments + 1).replace(0, np.nan),
        "LoanToAnnualIncome": x.LoanAmount / income,
        "LoanPaymentToIncome": x.MonthlyLoanPayment / monthly,
        "AssetsToIncome": x.TotalAssets / income,
        "LiabilitiesToAssets": x.TotalLiabilities / x.TotalAssets.replace(0, np.nan),
        "NetWorthToIncome": x.NetWorth / income,
        "SavingsToIncome": x.SavingsAccountBalance / income,
        "CheckingToIncome": x.CheckingAccountBalance / income,
        "SavingsToDebt": x.SavingsAccountBalance / (12 * x.MonthlyDebtPayments).replace(0, np.nan),
        "LiquidAssetsToDebt": (x.SavingsAccountBalance + x.CheckingAccountBalance)
            / x.TotalLiabilities.replace(0, np.nan),
        "CreditHistoryPerAge": x.LengthOfCreditHistory / x.Age.replace(0, np.nan),
        "ExperiencePerAge": x.Experience / x.Age.replace(0, np.nan),
        "JobTenurePerExperience": x.JobTenure / x.Experience.replace(0, np.nan),
        "CreditLinesPerHistory": x.NumberOfOpenCreditLines / x.LengthOfCreditHistory.replace(0, np.nan),
        "InquiriesPerCreditLine": x.NumberOfCreditInquiries / (x.NumberOfOpenCreditLines + 1).replace(0, np.nan),
        "InterestRateRelative": x.InterestRate / x.BaseInterestRate.replace(0, np.nan),
        "CreditScorePerInquiry": x.CreditScore / (x.NumberOfCreditInquiries + 1).replace(0, np.nan),
    })


def extra_features(frame):
    steps = pd.DataFrame(threshold_features(frame, EXTRA_THRESHOLDS), index=frame.index)
    ratios = financial_ratios(frame)
    signed_log = np.sign(ratios) * np.log1p(ratios.abs())
    return pd.concat([steps, signed_log], axis=1).replace([np.inf, -np.inf], np.nan).fillna(0)

## Разметка веток на обучении

Начальное разбиение задаёт медиана `RiskScore`. В каждой ветке обучаю
`LinearRegression` на основных ступеньках. Затем отношу заявку к той
ветке, чей прогноз ближе к её фактическому `RiskScore`. Повторяю до
стабилизации разметки или до 50 итераций.

In [14]:
main_train = main_features(train)
main_test = main_features(test)


def assign_branches(steps, target):
    upper = target > np.median(target)
    for _ in range(50):
        low = LinearRegression().fit(steps.loc[~upper], target[~upper])
        high = LinearRegression().fit(steps.loc[upper], target[upper])
        low_pred = low.predict(steps)
        high_pred = high.predict(steps)
        updated = np.abs(target - high_pred) < np.abs(target - low_pred)
        if np.array_equal(upper, updated):
            break
        upper = updated
    return upper if high_pred.mean() >= low_pred.mean() else ~upper


with threadpool_limits(limits=2):
    upper = assign_branches(main_train, y)
print(f"Нижняя ветка: {(~upper).sum()}, верхняя ветка: {upper.sum()}")

Нижняя ветка: 4733, верхняя ветка: 4599


## Вес ветки по признакам заявки

Обучаю линейную регрессию предсказывать разметку $0/1$. Первый проход
использует все заявки. Следующие пять проходов уточняют границу на заявках,
для которых предыдущий индекс попал в $(0,1)$. В итоговом прогнозе обрезаю
индекс до $[0,1]$ и использую как вес $p$ верхней ветки.

In [15]:
EDUCATION_ORDER = {"High School": 0, "Associate": 1, "Bachelor": 2, "Master": 3, "Doctorate": 4}
CATEGORIES = ["MaritalStatus", "HomeOwnershipStatus", "LoanPurpose", "EmploymentStatus"]


def index_features(frame):
    numeric = frame.select_dtypes(include=np.number).drop(
        columns=["ID", "AnnualIncome", "BaseInterestRate"], errors="ignore"
    ).astype(float)
    numeric["AgeSquared"] = frame.Age.astype(float) ** 2
    numeric["ApplicationMonth"] = pd.to_datetime(frame.ApplicationDate).dt.month
    numeric["EducationOrder"] = frame.EducationLevel.map(EDUCATION_ORDER)
    categories = pd.get_dummies(frame[CATEGORIES], drop_first=True, dtype=float)
    return pd.concat([numeric, categories], axis=1)


index_train = index_features(train)
index_test = index_features(test).reindex(columns=index_train.columns, fill_value=0.0)
index_scaler = StandardScaler().fit(index_train)
z_train = index_scaler.transform(index_train)
z_test = index_scaler.transform(index_test)

transition = np.ones(len(train), dtype=bool)
with threadpool_limits(limits=2):
    for _ in range(6):
        branch_index = LinearRegression().fit(z_train[transition], upper[transition])
        index = branch_index.predict(z_train)
        transition = (index > 0) & (index < 1)
p = np.clip(branch_index.predict(z_test), 0, 1)

## Линейная регрессия в каждой ветке

Для дополнительных признаков вычисляю средние, масштабы и диапазоны на
обучающей выборке. На `test` обрезаю их значения до обучающих диапазонов
перед стандартизацией.

В каждой ветке минимизирую сумму квадратов ошибок и два штрафа. Первый
штрафует квадрат разности коэффициентов соседних порогов одного признака,
чтобы сгладить зависимость. Второй штрафует квадраты коэффициентов
дополнительных признаков.

Каждое слагаемое штрафа записываю как искусственную строку с ответом 0.
Например, строка с $(\sqrt{\lambda},-\sqrt{\lambda})$ в двух соседних
позициях добавляет ровно $\lambda(\beta_j-\beta_{j+1})^2$ к сумме квадратов.
Поэтому итоговое обучение выполняет обычный `LinearRegression`. Столбец
единиц задаёт свободный член. Диагональная добавка $10^{-8}$ сохраняет
численную стабилизацию исходного решения.

In [16]:
extra_train = extra_features(train)
extra_test = extra_features(test).clip(extra_train.min(), extra_train.max(), axis="columns")
extra_scaler = StandardScaler().fit(extra_train)
phi_train = np.column_stack([np.ones(len(train)), main_train, extra_scaler.transform(extra_train)])
phi_test = np.column_stack([np.ones(len(test)), main_test, extra_scaler.transform(extra_test)])

unit = np.eye(phi_train.shape[1])
difference_rows = []
start = 1
for thresholds in MAIN_THRESHOLDS.values():
    stop = start + len(thresholds)
    difference_rows.append(unit[start:stop - 1] - unit[start + 1:stop])
    start = stop

penalties = np.vstack([
    np.sqrt(len(train) * 100 / 6218) * np.vstack(difference_rows),
    np.sqrt(len(train) * 10_000 / 6218) * unit[1 + main_train.shape[1]:],
    1e-4 * unit,
])
zeros = np.zeros(len(penalties))

with threadpool_limits(limits=2):
    branch_models = [
        LinearRegression(fit_intercept=False).fit(
            np.vstack([phi_train[mask], penalties]), np.r_[y[mask], zeros]
        )
        for mask in (~upper, upper)
    ]
print(f"Признаков на ветку: {phi_train.shape[1] - 1}")

Признаков на ветку: 88


## Прогноз и сохранение CSV

Для каждой тестовой заявки считаю оба прогноза. Итоговое значение
$\hat y=(1-p)f_{\mathrm{low}}(X)+pf_{\mathrm{high}}(X)$ плавно переходит
между ветками при изменении веса $p$.

In [17]:
low_prediction, high_prediction = [model.predict(phi_test) for model in branch_models]
predictions = (1 - p) * low_prediction + p * high_prediction
submission = pd.DataFrame({"ID": test.ID, "RiskScore": predictions})

assert len(submission) == len(test) and submission.ID.is_unique
assert submission.ID.equals(test.ID) and np.isfinite(submission.RiskScore).all()
output = root / "answer.csv"
submission.to_csv(output, index=False)
print(f"Сохранено {len(submission)} прогнозов в {output.name}")
display(submission.head())

Сохранено 5000 прогнозов в answer.csv


,ID,RiskScore
0,0,32.894708
1,1,53.740389
2,2,29.989759
3,3,34.858445
4,4,33.165455
